# Очистка

In [1]:
import pandas as pd


pd.set_option("display.max_columns", None)

df = pd.read_csv('../data/raw/olympiad_results.csv')
df_raw = df.copy()

## Исправление 'region'

In [2]:
def display_region_values(region_series):
	print('Уникальные значения:', region_series.nunique())
	display(pd.DataFrame({
		'count': region_series.value_counts(),
		'fraction': region_series.value_counts(normalize=True).round(2)
	}).sort_index())


display_region_values(df_raw['region'])

Уникальные значения: 19


,count,fraction
region,,
Moskva,19,0.00
Воронеж,807,0.10
Екатеринбург,754,0.09
Казань,748,0.09
Краснодар,807,0.10
Москва,736,0.09
Новосибирск,791,0.10
Пермь,766,0.10
Ростов-на-Дону,804,0.10


Эталонные названия городов для исправления написания:
- Москва (`Moskva`, `г. Москва`, `москва`, `мск`)
- Казань (`г.Казань`, `казань`)
- Санкт-Петербург (`СПб`, `Санкт Петербург`, `спб`)

In [3]:
incorrect_map = {
	'Москва': ['Moskva', 'г. Москва', 'москва', 'мск'],
	'Казань': ['г.Казань', 'казань'],
	'Санкт-Петербург': ['СПб', 'Санкт Петербург', 'спб']
}
replace_map = {}
for [correct_name, incorrect_names] in incorrect_map.items():
	for name in incorrect_names:
		replace_map[name] = correct_name
df['region'] = df['region'].replace(replace_map)

display_region_values(df['region'])
display(df['region'].drop_duplicates()[lambda x: ~x.isin(replace_map.keys())])

Уникальные значения: 10


,count,fraction
region,,
Воронеж,807,0.10
Екатеринбург,754,0.09
Казань,821,0.10
Краснодар,807,0.10
Москва,832,0.10
Новосибирск,791,0.10
Пермь,766,0.10
Ростов-на-Дону,804,0.10
Санкт-Петербург,808,0.10


0           Краснодар
1      Ростов-на-Дону
2        Екатеринбург
3              Казань
4                 Уфа
6     Санкт-Петербург
9         Новосибирск
11            Воронеж
12              Пермь
18             Москва
Name: region, dtype: str

## Приведение к типам

In [4]:
df_raw.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 8037 entries, 0 to 8036
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          8037 non-null   str    
 1   region              8037 non-null   str    
 2   school_number       8037 non-null   int64  
 3   grade               8037 non-null   int64  
 4   start_time          8037 non-null   str    
 5   end_time            8037 non-null   str    
 6   Task_1_answer       7719 non-null   float64
 7   Task_1_correct      7719 non-null   float64
 8   Task_1_time_spent   7719 non-null   float64
 9   Task_2_answer       7696 non-null   float64
 10  Task_2_correct      7696 non-null   float64
 11  Task_2_time_spent   7696 non-null   float64
 12  Task_3_answer       7722 non-null   float64
 13  Task_3_correct      7722 non-null   float64
 14  Task_3_time_spent   7722 non-null   float64
 15  Task_4_answer       7727 non-null   float64
 16  Task_4_correct   

In [5]:
df['start_time'] = pd.to_datetime(df['start_time'], format='%Y-%m-%d %H:%M:%S')
df['end_time']= pd.to_datetime(df['end_time'], format='%Y-%m-%d %H:%M:%S')

category_columns = ['region', 'device_type']
task_n_answer_columns = df.columns[df.columns.str.match(r"^Task_\d+_answer$")].tolist()
for category in category_columns + task_n_answer_columns:
	df[category] = df[category].astype('category')
df['grade'] = df['grade'].astype(pd.CategoricalDtype(categories=sorted(df['grade'].unique()), ordered=True))

task_n_correct_columns = df.columns[df.columns.str.match(r"^Task_\d+_correct$")].to_list()
for task_n_correct in task_n_correct_columns:
	df[task_n_correct] = df[task_n_correct].astype('Int8')

task_n_time_spent_columns = df.columns[df.columns.str.match(r"^Task_\d+_time_spent$")].to_list()
for task_n_time_spent in task_n_time_spent_columns:
	df[task_n_time_spent] = df[task_n_time_spent].astype('Int16')

df['total_score'] = df['total_score'].astype('int')

df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 8037 entries, 0 to 8036
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   student_id          8037 non-null   str           
 1   region              8037 non-null   category      
 2   school_number       8037 non-null   int64         
 3   grade               8037 non-null   category      
 4   start_time          8037 non-null   datetime64[us]
 5   end_time            8037 non-null   datetime64[us]
 6   Task_1_answer       7719 non-null   category      
 7   Task_1_correct      7719 non-null   Int8          
 8   Task_1_time_spent   7719 non-null   Int16         
 9   Task_2_answer       7696 non-null   category      
 10  Task_2_correct      7696 non-null   Int8          
 11  Task_2_time_spent   7696 non-null   Int16         
 12  Task_3_answer       7722 non-null   category      
 13  Task_3_correct      7722 non-null   Int8          
 14  Tas

## Очистка дубликатов

In [6]:
print('Полных дупликатов:', df_raw.duplicated().sum())

duplicates = df_raw[df_raw.duplicated(keep=False)].sort_values('student_id')
print('Всего строк:', len(duplicates))
display(duplicates)

Полных дупликатов: 37
Всего строк: 74


,student_id,region,school_number,grade,start_time,end_time,Task_1_answer,Task_1_correct,Task_1_time_spent,Task_2_answer,Task_2_correct,Task_2_time_spent,Task_3_answer,Task_3_correct,Task_3_time_spent,Task_4_answer,Task_4_correct,Task_4_time_spent,Task_5_answer,Task_5_correct,Task_5_time_spent,Task_6_answer,Task_6_correct,Task_6_time_spent,Task_7_answer,Task_7_correct,Task_7_time_spent,Task_8_answer,Task_8_correct,Task_8_time_spent,Task_9_answer,Task_9_correct,Task_9_time_spent,Task_10_answer,Task_10_correct,Task_10_time_spent,Task_11_answer,Task_11_correct,Task_11_time_spent,Task_12_answer,Task_12_correct,Task_12_time_spent,total_score,device_type,attempts_count,disconnects
234,S200234,Санкт-Петербург,35,9,2026-04-10 10:41:00,2026-04-10 10:55:19,2.0,0.0,55.0,3.0,0.0,68.0,2.0,0.0,81.0,1.0,1.0,93.0,1.0,0.0,67.0,3.0,1.0,59.0,1.0,0.0,64.0,1.0,0.0,88.0,2.0,0.0,98.0,2.0,0.0,43.0,4.0,0.0,68.0,1.0,1.0,75.0,3,mobile,1,0
235,S200234,Санкт-Петербург,35,9,2026-04-10 10:41:00,2026-04-10 10:55:19,2.0,0.0,55.0,3.0,0.0,68.0,2.0,0.0,81.0,1.0,1.0,93.0,1.0,0.0,67.0,3.0,1.0,59.0,1.0,0.0,64.0,1.0,0.0,88.0,2.0,0.0,98.0,2.0,0.0,43.0,4.0,0.0,68.0,1.0,1.0,75.0,3,mobile,1,0
641,S200640,Пермь,48,11,2026-04-10 10:37:00,2026-04-10 10:54:46,2.0,0.0,49.0,4.0,0.0,70.0,2.0,0.0,113.0,1.0,1.0,113.0,3.0,1.0,75.0,3.0,1.0,88.0,2.0,1.0,103.0,1.0,1.0,69.0,4.0,0.0,103.0,3.0,0.0,124.0,2.0,1.0,104.0,2.0,1.0,55.0,7,desktop,1,0
642,S200640,Пермь,48,11,2026-04-10 10:37:00,2026-04-10 10:54:46,2.0,0.0,49.0,4.0,0.0,70.0,2.0,0.0,113.0,1.0,1.0,113.0,3.0,1.0,75.0,3.0,1.0,88.0,2.0,1.0,103.0,1.0,1.0,69.0,4.0,0.0,103.0,3.0,0.0,124.0,2.0,1.0,104.0,2.0,1.0,55.0,7,desktop,1,0
963,S200961,Санкт-Петербург,27,7,2026-04-10 09:25:00,2026-04-10 09:40:14,3.0,1.0,106.0,1.0,0.0,31.0,4.0,0.0,26.0,3.0,1.0,79.0,2.0,0.0,99.0,3.0,0.0,92.0,2.0,1.0,109.0,4.0,1.0,78.0,4.0,0.0,59.0,2.0,0.0,89.0,1.0,1.0,103.0,3.0,0.0,43.0,5,desktop,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7841,S207806,Пермь,44,11,2026-04-10 09:07:00,2026-04-10 09:21:41,3.0,0.0,95.0,2.0,1.0,90.0,2.0,0.0,81.0,2.0,1.0,85.0,1.0,1.0,131.0,2.0,1.0,56.0,1.0,0.0,68.0,4.0,1.0,70.0,4.0,1.0,88.0,4.0,0.0,95.0,NaN,NaN,NaN,4.0,0.0,22.0,6,desktop,2,0
7981,S207946,Воронеж,5,10,2026-04-10 10:29:00,2026-04-10 10:41:54,4.0,0.0,55.0,3.0,0.0,39.0,3.0,1.0,74.0,4.0,0.0,103.0,3.0,0.0,114.0,3.0,0.0,76.0,1.0,0.0,99.0,3.0,0.0,74.0,2.0,0.0,65.0,1.0,0.0,57.0,2.0,1.0,18.0,NaN,NaN,NaN,2,mobile,2,0
7982,S207946,Воронеж,5,10,2026-04-10 10:29:00,2026-04-10 10:41:54,4.0,0.0,55.0,3.0,0.0,39.0,3.0,1.0,74.0,4.0,0.0,103.0,3.0,0.0,114.0,3.0,0.0,76.0,1.0,0.0,99.0,3.0,0.0,74.0,2.0,0.0,65.0,1.0,0.0,57.0,2.0,1.0,18.0,NaN,NaN,NaN,2,mobile,2,0
8028,S207992,Ростов-на-Дону,31,5,2026-04-10 09:50:00,2026-04-10 10:03:01,4.0,0.0,90.0,1.0,0.0,30.0,3.0,0.0,56.0,4.0,0.0,96.0,2.0,1.0,52.0,1.0,0.0,67.0,1.0,0.0,54.0,4.0,0.0,77.0,1.0,0.0,87.0,2.0,0.0,70.0,3.0,0.0,34.0,1.0,0.0,68.0,1,desktop,1,0


In [7]:
df = df.drop_duplicates()

print('Полных дупликатов:', df.duplicated().sum())

Полных дупликатов: 0


# Итог

In [8]:
df_raw.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 8037 entries, 0 to 8036
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          8037 non-null   str    
 1   region              8037 non-null   str    
 2   school_number       8037 non-null   int64  
 3   grade               8037 non-null   int64  
 4   start_time          8037 non-null   str    
 5   end_time            8037 non-null   str    
 6   Task_1_answer       7719 non-null   float64
 7   Task_1_correct      7719 non-null   float64
 8   Task_1_time_spent   7719 non-null   float64
 9   Task_2_answer       7696 non-null   float64
 10  Task_2_correct      7696 non-null   float64
 11  Task_2_time_spent   7696 non-null   float64
 12  Task_3_answer       7722 non-null   float64
 13  Task_3_correct      7722 non-null   float64
 14  Task_3_time_spent   7722 non-null   float64
 15  Task_4_answer       7727 non-null   float64
 16  Task_4_correct   

In [9]:
df = df.reset_index(drop=True)
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   student_id          8000 non-null   str           
 1   region              8000 non-null   category      
 2   school_number       8000 non-null   int64         
 3   grade               8000 non-null   category      
 4   start_time          8000 non-null   datetime64[us]
 5   end_time            8000 non-null   datetime64[us]
 6   Task_1_answer       7682 non-null   category      
 7   Task_1_correct      7682 non-null   Int8          
 8   Task_1_time_spent   7682 non-null   Int16         
 9   Task_2_answer       7660 non-null   category      
 10  Task_2_correct      7660 non-null   Int8          
 11  Task_2_time_spent   7660 non-null   Int16         
 12  Task_3_answer       7686 non-null   category      
 13  Task_3_correct      7686 non-null   Int8          
 14  Tas

In [10]:
df = df.reset_index(drop=True)
df.to_parquet('olympiad_results_clean.parquet')